In [2]:
rows, cols = 5, 5
initial_state = (0, 0)
goal_state = (0, 3)

# Define special edges between cells
# Each edge may be a wall (blocked), a bridge, or a river (with rewards).
edges = {
    ((1, 2), (1, 3)): {"type": "bridge", "reward": 9},
    ((0, 4), (1, 4)): {"type": "river", "reward": 3},
    ((0, 0), (0, 1)): {"type": "wall"},
    ((0, 1), (1, 1)): {"type": "wall"},
    ((0, 2), (1, 2)): {"type": "wall"},
    ((1, 1), (1, 2)): {"type": "wall"},
    ((1, 3), (1, 4)): {"type": "wall"},
    ((1, 3), (2, 3)): {"type": "wall"},
    ((2, 0), (2, 1)): {"type": "wall"},
    ((2, 1), (2, 2)): {"type": "wall"},
    ((2, 0), (3, 0)): {"type": "wall"},
    ((3, 2), (3, 3)): {"type": "wall"},
    ((3, 3), (3, 4)): {"type": "wall"},
    ((2, 4), (3, 4)): {"type": "wall"},
    ((3, 1), (4, 1)): {"type": "wall"},
}

Emtiyaz = 0       # Total reward (or cost)
seen = []         # Order of nodes actually visited
path = []         # Current path towards the goal
visited = set()   # To avoid revisiting the same nodes

def move(current):
    global Emtiyaz, seen, path, visited

    # Check if goal is reached
    if current == goal_state:
        print("Goal Reached!")
        return

    a, b = current

    # Define possible moves (4 directions)
    directions = {
        "left":  (a, b - 1),
        "down":  (a + 1, b),
        "right": (a, b + 1),
        "up":    (a - 1, b)
    }

    # Compute A* score (f = g + h) for each neighbor
    rewards = {}
    for d, pos in directions.items():
        if 0 <= pos[0] < rows and 0 <= pos[1] < cols:
            if not isThere_wall(current, pos) and pos not in visited:
                rewards[d] = cal_aStar_reward(pos, goal_state, path + [current])
            else:
                rewards[d] = float("inf")  # invalid move
        else:
            rewards[d] = float("inf")      # outside grid

    # Dead-end: all neighbors are invalid → backtrack
    if all(r == float("inf") for r in rewards.values()):
        if current not in seen:
            seen.append(current)
        if len(path) > 1:
            cur = path.pop()
            prev = path[-1]
            Emtiyaz -= get_reward(prev, cur)  # undo reward of last step
            move(prev)                        # backtrack
        return

    # Select direction with minimum f-value (A*) with fixed tie-break priority
    min_dirs = min_reward(rewards)
    priority = ["left", "down", "right", "up"]  # tie-break order
    for d in priority:
        if d in min_dirs:
            nextMove = d
            break

    next_pos = directions[nextMove]
    Emtiyaz += get_reward(current, next_pos)  # update reward

    visited.add(next_pos)
    if current not in path:
        path.append(current)
    seen.append(next_pos)
    path.append(next_pos)

    move(next_pos)  # recursive call to continue search

# Check if there is a wall between two positions
def isThere_wall(current, nextstep):
    edge = edges.get((current, nextstep)) or edges.get((nextstep, current))
    return edge and edge["type"] == "wall"

# Return all directions with the minimum f-value
def min_reward(rewards):
    valid = {k: v for k, v in rewards.items() if v < float("inf")}
    m = min(valid.values())
    return [d for d, v in valid.items() if v == m]

# Get reward of moving between two cells
def get_reward(current_pos, next_pos):
    edge = edges.get((current_pos, next_pos)) or edges.get((next_pos, current_pos))
    if edge:
        if edge["type"] == "bridge":
            return 9
        elif edge["type"] == "river":
            return 3
    return 1  # normal move

# Heuristic: Manhattan distance to the goal
def manhattan_distance(pos, goal):
    return abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])

# Compute g(n): total cost of a path so far
def cal_gn(path):
    total_cost = 0
    for i in range(len(path) - 1):
        total_cost += get_reward(path[i], path[i + 1])
    return total_cost

# Compute f(n) = g(n) + h(n) for A*
def cal_aStar_reward(current_pos, goal_pos, path):
    g = cal_gn(path + [current_pos])
    h = manhattan_distance(current_pos, goal_pos)
    return g + h

def main():
    start = (0, 0)
    visited.add(start)
    seen.append(start)
    path.append(start)
    move(start)  # start search
    print("Seen:", seen)
    print("Path:", path)
    print("Emtiyaz:", Emtiyaz)

main()

Goal Reached!
Seen: [(0, 0), (1, 0), (1, 1), (2, 1), (3, 1), (3, 2), (2, 2), (2, 3), (3, 3), (4, 3), (4, 2), (4, 1), (4, 0), (3, 0), (4, 4), (3, 4), (2, 4), (1, 4), (0, 4), (0, 3)]
Path: [(0, 0), (1, 0), (1, 1), (2, 1), (3, 1), (3, 2), (2, 2), (2, 3), (2, 4), (1, 4), (0, 4), (0, 3)]
Emtiyaz: 13
